# Databricks notebook source
# MAGIC %md
# MAGIC # 03 Gold Training Metrics
# MAGIC
# MAGIC This notebook reads the Silver activity summary table and creates analytics-ready Gold tables.


In [0]:
# COMMAND ----------

from pyspark.sql.functions import col, round, to_date, count, sum, avg, max, current_timestamp

In [0]:
# COMMAND ----------

silver_table_name = "athlete_training_lakehouse.silver_activity_summary"
gold_table_name = "athlete_training_lakehouse.gold_athlete_training_summary"

In [0]:
# COMMAND ----------

silver_df = spark.table(silver_table_name)

display(silver_df)

In [0]:
# COMMAND ----------

gold_df = (
    silver_df
        .withColumn("activity_date", to_date(col("activity_start_time"), "M/d/yyyy H:mm"))
        .withColumn(
            "avg_pace_min_per_mile",
            round(col("duration_minutes") / col("distance_miles"), 2)
        )
        .select(
            "activity_id",
            "athlete_id",
            "athlete_name",
            "activity_date",
            "distance_miles",
            "duration_minutes",
            "elevation_gain_feet",
            "avg_pace_min_per_mile"
        )
        .withColumn("gold_updated_at", current_timestamp())
)


In [0]:
# COMMAND ----------

(
    gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table_name)
)


In [0]:
# COMMAND ----------

display(spark.table(gold_table_name))


In [0]:
# COMMAND ----------

row_count = spark.table(gold_table_name).count()

print(f"Gold table created successfully: {gold_table_name}")
print(f"Rows loaded: {row_count}")

In [0]:
# COMMAND ----------

from pyspark.sql.functions import count, sum, avg, max, min

gold_summary_table_name = "athlete_training_lakehouse.gold_athlete_summary"

In [0]:
# COMMAND ----------

gold_activity_df = spark.table(gold_table_name)

gold_athlete_summary_df = (
    gold_activity_df
        .groupBy("athlete_id", "athlete_name")
        .agg(
            count("activity_id").alias("total_runs"),
            round(sum("distance_miles"), 2).alias("total_miles"),
            round(avg("distance_miles"), 2).alias("avg_distance_miles"),
            round(avg("avg_pace_min_per_mile"), 2).alias("avg_pace_min_per_mile"),
            round(max("distance_miles"), 2).alias("longest_run_miles"),
            round(sum("elevation_gain_feet"), 0).alias("total_elevation_gain_feet"),
            min("activity_date").alias("first_activity_date"),
            max("activity_date").alias("latest_activity_date"),
            current_timestamp().alias("gold_updated_at")
        )
)

In [0]:
# COMMAND ----------

(
    gold_athlete_summary_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_summary_table_name)
)

In [0]:
# COMMAND ----------

display(spark.table(gold_summary_table_name))

In [0]:
# COMMAND ----------

row_count = spark.table(gold_summary_table_name).count()

print(f"Gold athlete summary table created successfully: {gold_summary_table_name}")
print(f"Rows loaded: {row_count}")